# Tricks and tips for efficient connectivity analyses
Copyright (c) 2025 Open Brain Institute

Authors: Michael W. Reimann

Last modified: 10.2025

## Summary
This is an instructive notebook that outlines scipy / numpy / pandas techniques for efficient analysis of neuronal connectivity in large models or large biological datasets.

Once the data to be analyzed contains more than a few tens of thousands of neurons, an enormous speedup can be gained by moving from naive to more improved coding schemes. Additionally, such schemes allow the analysis of even larger models without running into resource limitations (out-of-memory errors).

# Find and download a circuit model
After importing the required packages, we authenticate with the OBI platform to get access to the circuit models stored.
Below, please follow the instructions to authenticate and select one of your projects.

In [ ]:
from entitysdk import Client, models
from entitysdk._server_schemas import AssetLabel
from obi_auth import get_token
import os
import time
import json
import conntility
import numpy
import tqdm

from datetime import datetime
from obi_notebook import get_projects
from obi_notebook import get_entities
from obi_notebook.get_environment import get_environment

token = get_token(environment=get_environment(), auth_mode="daf")
project_context = get_projects.get_projects(token)

## Selection of a circuit

Next, we find and select one of the microcircuit-scale models stored in the platform.
You can either find one using the `Data` section of the platform and copy-paste its ID below. Alternatively, a widget for the selection will be displayed.

In [ ]:
client = Client(environment=get_environment(), project_context=project_context, token_manager=token)

# Optional: Download using unique ID
entity_ID = "<CIRCUIT-ID>"  # <<< FILL IN UNIQUE CIRCUIT ID HERE


if entity_ID != "<CIRCUIT-ID>":
    circuit_ids = [entity_ID]
else:
# Alternative: Select from a table of entities
    circuit_ids = []
    circuit_ids = get_entities.get_entities("circuit", token, circuit_ids,
                                            project_context=project_context,
                                            multi_select=False, exclude_scales=["single"],
                                            show_pages=True, page_size=12,
                                            default_scale="microcircuit", add_columns=["subject.name"])

## Download the connectivity matrix of the selected circuit.

Next, we download the connectivity of the selected model. We download it in a format we developed for analyses of the anatomy of connectivity. This format strips out most the data required to simulate the circuit, but keeps the data that describes the anatomy. Additionally, the format supports many useful functions for connectivity analyses.

In principle, we could perform the analyses below also from the representation of the circuit model that is used in simulations (SONATA format). But this way it is easier.

In [ ]:
# Ask platform for information about the selected circuit
circ_entity = client.get_entity(circuit_ids[0], entity_type=models.Circuit)
# Find the `circuit connectivity matrices` within the files stored for the circuit
matrix_assets = [asset for asset in circ_entity.assets if asset.label==AssetLabel.circuit_connectivity_matrices]
if len(matrix_assets) == 0:
    print("The selected circuit has no registered connectivity matrices. Please select a different one!")

# Download the `circuit_connectivity_matrices`
dl_path = "circuit_connection_matrices"
matrix_files = client.download_directory(entity_id=circuit_ids[0], asset_id=matrix_assets[0],
                                         entity_type=models.Circuit, output_path=dl_path,
                                         ignore_directory_name=True)
# The `matrix_config` is a json files tells us which types of connectivity matrices we just downloaded
matrix_config = [fn for fn in matrix_files if os.path.splitext(fn)[1] == ".json"]  
with open(matrix_config[0], "r") as fid:
    matrix_config = json.load(fid)
# We simply use the first such matrix
matrix_path = list(list(matrix_config.values())[0].values())[0]["path"]

# We load the matrix using the `conntility` package.
M = conntility.ConnectivityMatrix.from_h5(os.path.join(dl_path, matrix_path))  

In [ ]:
print(f"We just loaded the intrinsic connectivity between {len(M)} neurons!")

## Select a submatrix
The purpose of this notebook is to demonstrate techniques that enable analyses of networks of hundreds of thousands of neurons.
But to do so, we also demonstrate some less optimized techniques for contrast. Therefore, we select a random subcircuit of up to 5000 neurons for demonstration.

In [ ]:
n_to_select = 5000
random_subset = numpy.random.choice(M.gids, numpy.minimum(len(M), n_to_select), replace=False)
S = M.subpopulation(random_subset)

print(f"Sub-circuit has {len(S)} neurons!")

# Example: Connection probability within 150 um

This is a common analysis: What is the mean connection probability for pairs of neurons within 150 um (or any other similar cutoff)?

## Simple approach
A simple approach is the following: We consider the _adjacency matrix_ of the circuit, i.e., a numpy.array where the entry at i, j is `True` iff a connection exists from neuron i to j. We also consider the _distance matrix_, i.e., a numpy.array where the entry at i, j is the distance between the somata of neurons i and j. From the distance matrix we create a mask that is `True` for all pairs within the distance cutoff. We use the mask with the adjacency matrix to calculate its mean value for the non-masked elements.

### Adjacency matrix
Getting the adjacency matrix is easy. It is a property of `S`. There is just one small hickup: by default, its elements are not `True` and `False`, but an integer that specified the _number of synapses_ from the source to the target neuron, with 0 indicating no connection. We solve this by simply casting the adjacency matrix to the `bool` data type.

In [ ]:
adjacency_matrix = S.array.astype(bool)
print(adjacency_matrix)

### Distance matrix
This one is more complicated. 
As a first approach we might simply iterate over all pairs of neurons, one by one, calculate the corresponding distance and putting the result into the correct position of the matrix.

This works, but it will be very slow.

The `S` object contains `vertices` that represent the neurons of the circuit. They are associated with various properties, such as neuron types. They are also associated with the `x` `y` and `z` coordinates (in um), which we can use to calculate their distances.

### NOTE:
*Feel free to interrupt the execution of the cell below, and continue with the next cell. Because it will take very long*

In [ ]:
xyz = ["x", "y", "z"]

# Allocate matrix of the appropriate size. 
D = -numpy.ones((len(S), len(S)))

# Iterate over source and target neurons. Calculate their distance
for i, coord_src in tqdm.tqdm(S.vertices[xyz].iterrows()):
    for j, coord_tgt in S.vertices[xyz].iterrows():
        D[i, j] = numpy.linalg.norm(coord_src - coord_tgt)

It takes very long because `for` loops in python tend to be very slow.
Nested for loops, such as the one above, should be avoided even more.

We can improve the situation by removing the inner `for` loop and instead calculate the distance from a source neuron to all other neurons in a single line. 

Note that this still iterates over all target neurons, but it does this inside the code belonging to the `pandas` package, where it is much faster. See below:

In [ ]:
D = -numpy.ones((len(S), len(S)))

t_start = time.time()
for i, coord_src in tqdm.tqdm(S.vertices[xyz].iterrows()):
    D[i, :] = numpy.linalg.norm(coord_src - S.vertices[xyz], axis=1)

print(time.time() - t_start)

It is even faster to avoid the loop completely and instead get all pairwise distances in a single line.
For this we can use some `numpy` functionality to reshape the array of neuron x,y,z-coordinates in a very specific fashion.

We first get a 2d numpy array of neuron positions, where neurons are listed along the first dimension and the three axes along the second.
Reshaping this to `(-1, 1, 3)` keeps neurons along the first dimension, while `(1, -1, 3)` pushes them into the second dimension. In both cases, x/y/z are turned into the _third_ dimension. Subtracting the two resulting arrays creates the differences for _all pairs_ and puts them along the first two dimensions. The third dimension remains x/y/z.

This is exactly what we need to calculate all pairwise distances using `numpy.linalg.norm`

In [ ]:
t_start = time.time()

xyz_np = S.vertices[xyz].to_numpy()
pw_dist = xyz_np.reshape((-1, 1, 3)) - xyz_np.reshape((1, -1, 3))
D = numpy.linalg.norm(pw_dist, axis=-1)

print(time.time() - t_start)

The above is already very usable, but one thing is even faster: Using a specialized function.

The `scipy` package has a function specifically for pairwise distances in euclidean space. See it in action below.

In [ ]:
from scipy.spatial.distance import pdist, squareform

t_start = time.time()
D = squareform(pdist(S.vertices[xyz]))

print(time.time() - t_start)

With the distance matrix `D` in hand, it is simple to calculate the mean connection probability within a given distance.

(Question: Why do we also test `D > 0` to build the mask below?)

In [ ]:
max_distance = 150

con_prob = adjacency_matrix[(D <= max_distance) & (D > 0)].mean()
print(f"Mean connection probability within {max_distance} um is {con_prob * 100}%")

## Using a sparse representation instead
The above gives us very rapid estimates of connection probabilities for circuits up to tens of thousands of neurons easily.

But once we go to even larger models of hundreds of thousands of neurons we begin to encounter problems: The adjacancy and distance matrices grow with the square of the number of neurons and get too large to fit into computer memory. Instead of simply getting a larger computer we can try a more efficient strategy.

In neuronal circuitry, connectivity tends to be very sparse. That means that less than 1% of all pairs are actually connected to each other. When we built the full adjacency and distance matrices, the vast majority of pairs were unconnected and thus not of interest to us. Arguably, it was a waste of resources to consider them.

Instead, we try to calculate the connection probability within a given distance *without considering all pairs of neurons*. How is that possible?

Once again, for the adjancency matrix, this is easy. `S` does not only provide functionality to represent the adjacency matrix as a numpy array, but also as a scipy.sparse matrix. Instead of values for all pairs, it represents the *coordinates* of the non-zero elements and their values. For very sparse matrices, such as neuronal connectivity, this is much more efficient.

In [ ]:
sp_adjacency = S.matrix.astype(bool)

print(sp_adjacency.row) # Row indices of non-zero elements, i.e., of connected pairs
print(sp_adjacency.col) # Column indices of non-zero elements.
print(sp_adjacency.data) # The values of non-zero elements. Will all be `True`

Since `row` and `col` are the indices of non-zero elements of the adjacency matrix they are also the indices of source and target neurons of connections.

This allows us to easily build a numpy array where each row corresponds to the source neuron of a *connection* and the columns to x/y/z. Similarly for the target neuron of a connection. Subtracting the two and calculating the `linalg.norm` yields the distances between source and target neuron of each connection.

In [ ]:
xyz_np = S.vertices[xyz].to_numpy() # Shape: number of neurons X 3

xyz_source = xyz_np[sp_adjacency.row] # Shape: number of connections x 3
xyz_target = xyz_np[sp_adjacency.col] # Shape: number of connections x 3

con_dists = numpy.linalg.norm(xyz_source - xyz_target, axis=1)

print(f"The mean distance between connected neuron pairs is {con_dists.mean()} um")

The arrays `xyz_source` and `xyz_target` above have one row per connections and are thus much larger than `xyz_np`, which only has one row per neuron. However, they are still vastly smaller than the dense adjacency matrices we used before, the number of connections is still small compared to the number of pairs.

Below is an equivalent implementation of the same principle using instead the `edge_associated_vertex_properties` function of `S`. It returns a pandas DataFrame with one row per connection and two columns: The first column has the value of a neuron property for the source neuron of the connection, the second the value for the target neuron. Iterating over "x", "y" and "z" coordinates allows us to assemble the distances as well.

In [ ]:
con_dists = numpy.zeros(len(S.edges))
for coord in ["x", "y", "z"]:
    deltas = S.edge_associated_vertex_properties(coord)
    con_dists += ((deltas["col"] - deltas["row"])) ** 2
con_dists = numpy.sqrt(con_dists)

print(f"The mean distance between connected neuron pairs is {con_dists.mean()} um")

This allows us to count the number of _connected_ neurons within the distance cutoff

In [ ]:
n_connected_within = numpy.sum(con_dists < max_distance)

print(f"There are {n_connected_within} connected pairs within {max_distance} um")

### Number of pairs within a given distance

To turn the number of connected pairs within the distance cutoff into a connection probability we need a second measure: The number of pairs of neurons within the distance that could potentially have been connected. That is, the number of pairs within the distance. How do we calculate that without assembling the full distance matrix?

Once again, we do this by using specialized functionality from `scipy`. In this case, a `KDTree`. It allows rapid lookup of which points are within a given maximum distance of a different (or the same) set of points. We simply use the x/y/z coordinates of neurons for the points.

In [ ]:
from scipy.spatial import KDTree

kdtree = KDTree(S.vertices[xyz])

#  For each neuron: Indices of neurons within max_distance
query_result = kdtree.query_ball_tree(kdtree, max_distance)
#  Using `len` to get the number of neurons within max distance. 
#  Subtract 1 because the neuron itself will always be in range (distance is 0).
n_within_per_neuron = [len(_x) - 1 for _x in query_result]
n_pairs_within = numpy.sum(n_within_per_neuron)

print(f"There are {n_pairs_within} pairs of neurons within {max_distance} um")
print(f"This yields a connection probability of {100*n_connected_within/n_pairs_within}%")

Note that the KDTree does not just provide the number of pairs within a given distance, but also the indices of all those pairs. This is incredibly useful for many analyses.

### Really big toy example
To demonstrate the utility, we create some fake connectivity and neuron locations for 250,000 neurons, a much larger example.

Through the use of sparse representations and KDTrees we can still easily calculate connection probability within 150 um. 
If you were to try that using a dense representation, you would most likely run out of memory.

In [ ]:
# Generate toy model
from scipy import sparse
n_neurons = 250000  # number of neurons.
tgt_p = 1E-3  # Target connection probability. This value is realistic for 250k neuron.

# Generate a random sparse matrix
n_connections = int(n_neurons * n_neurons * tgt_p)
row = numpy.random.choice(n_neurons, n_connections)
col = numpy.random.choice(n_neurons, n_connections)
data = numpy.ones_like(row, dtype=bool)

# Note: this matrix will have autapses, i.e., connections from a neuron to itself.
# Usually we would want to avoid that, but for this technical demo we don't care.
rnd_matrix = sparse.coo_matrix((data, (row, col)), shape=(n_neurons, n_neurons))
# Random locations for the neurons. We fill them into a 4mm X 4mm X 4mm cube.
rnd_xyz = numpy.random.rand(n_neurons, 3) * 4000

In [ ]:
# CONNECTED
xyz_source = rnd_xyz[rnd_matrix.row] # Shape: number of connections x 3
xyz_target = rnd_xyz[rnd_matrix.col] # Shape: number of connections x 3

con_dists = numpy.linalg.norm(xyz_source - xyz_target, axis=1)
n_connected_within = numpy.sum(con_dists <= max_distance)

kdtree = KDTree(rnd_xyz)

query_result = kdtree.query_ball_tree(kdtree, max_distance)
n_within_per_neuron = [len(_x) - 1 for _x in query_result]
n_pairs_within = numpy.sum(n_within_per_neuron)

print(f"Connection probability is {100*n_connected_within/n_pairs_within}%")


### Notes
All of this relies on connectivity being sparse, which most neuronal circuits are.

Additionally, the larger you select your distance cutoff, the less advantageous the sparse representation gets. Past a certain point it will be faster to go back to a dense representation, but assemble and analyze the distance and adjacency matrices row by row. 

# Simple motif search

Another type of analysis is the lookup of simple connectivity _motifs_. As before, this can be naively implemented by iterating over all neurons in nested loops. But many of them can be implemented much more efficiently using various forms of matrix multiplications of the sparse representation of the adjacency matrix.

## Reciprocal connections
The simplest motif is a _reciprocal connection_, i.e., a pair of neurons (i, j) where connections exist in both directions, from i to j and from j to i. 

To get the matrix of reciprocal connections we simply employ element-wise multiplication of the matrix with its transpose. We still use the sparse representation to avoid waste of resources.

In [ ]:
t_start = time.time()
sp_adjacency = S.matrix

# We convert from a "sparse matrix" to a "sparse array" to get access to element-wise multiplication
# instead of matrix multiplication 
sp_adjacency_arr = sparse.coo_array(sp_adjacency)
# Element-wise multiplication
reciprocal_adjacency = sp_adjacency_arr * sp_adjacency_arr.transpose()

# `.nnz` => number of nonzero elements, i.e. connections
print(f"There are {reciprocal_adjacency.nnz} reciprocal connections")
print(time.time() - t_start)

Below we compare this to the alternative implementation using dense representations of connectivity.

We see that sparse is faster, although not by much. However note once again that the dense representation will run out of memory for larger examples faster.

In [ ]:
# Dense alternative
t_start = time.time()

dense_adjacency_arr = S.array.astype(bool)
# Element-wise multiplication
reciprocal_adjacency = dense_adjacency_arr * dense_adjacency_arr.transpose()

print(f"There are {reciprocal_adjacency.sum()} reciprocal connections")
print(time.time() - t_start)

## Some triplet motifs
Some triplet motifs can be efficiently counted using "proper" matrix multiplication, i.e., not element-wise.

A triplet motif is any specific configuration of connectivity between three neurons. 

To illustrate, let's work through an example:
What happens if we perform matrix (non-element-wise!) multiplication of an an adjacency matrix with itself? In the output, the entry at location (i, j) will be assembled as follows: Take the `i`th row of the adjacency matrix and its `j`th column; then count the number of entries that are nonzero in both of the resulting vectors. Interpreted as a motif, this is the number of neurons `k` that simultaneously receive input from `i` and send output to `j`:

`i` --> `[k]` --> `j`

In other words, this counts the number of paths of length 2 from i to j.

In [ ]:
sp_adjacency = S.matrix.astype(bool).astype(int)

ikj_count = sp_adjacency * sp_adjacency  # NOT element-wise multiplication!
print(f"Mean number of paths of length 2 between a pair: {ikj_count.mean()}")

Note: Above we converted the adjacency matrix to data type int. If we forgo that, then `sp_adjancency` will also be of type `bool`, just like the adjacency matrix. An entry will be `True` iff any path of length 2 exists. This can also be useful.

In [ ]:
sp_adjacency = S.matrix.astype(bool)

ikj_count = sp_adjacency * sp_adjacency  # NOT element-wise multiplication!
print(f"Probability that a paths of length 2 exists between a pair: {ikj_count.mean()}")

While the overall connectivity is very sparse, you only need to take a second step to be able to reach around half of the neurons!


The above only counts the number of `k` between `i` and `j`, making no assertion on connectivity between `i` and `j`. 

Let's try to count the following motif:

`i` --> `k` --> `j` --> `i`

That is, we also require a connection from `j` to `i` turning the entire motif into a cycle of length 3. We can do that by simply element-wise multiplying `ikj_count` with the transpose of the adjacency matrix. That multiplication simply does the following: If the connection from `j` to `i` does not exist, then the entry `(i, j)` of the transposed adjacency matrix is zero and the corresponding entry of `ikj_counts` is set to zero.

In [ ]:
sp_adjacency = S.matrix.astype(bool).astype(int)

ikj_count = sp_adjacency * sp_adjacency

# Once again: Convert to sparse arrays to force element-wise multiplication
ikj_count_arr = sparse.coo_array(ikj_count)
sp_adjacency_arr = sparse.coo_array(sp_adjacency)

n_3_cycles = (ikj_count_arr * sp_adjacency_arr.transpose()).sum()

print(f"Number of 3-cycles: {n_3_cycles}")

What happens if we do not transpose `sp_adjacency_arr`? In that case, we considere the following motif instead:

      /--> j
`i` --> `k` --> `j`

This motif is called a directed simplex of dimension 2. We can count them

In [ ]:
sp_adjacency = S.matrix.astype(bool).astype(int)

ikj_count = sp_adjacency * sp_adjacency

ikj_count_arr = sparse.coo_array(ikj_count)
sp_adjacency_arr = sparse.coo_array(sp_adjacency)

n_3_simplices = (ikj_count_arr * sp_adjacency_arr).sum()

print(f"Number of 3-simplices: {n_3_simplices}")

We see that there are many more simplices than cycles. This is a known property of neuronal circuits!

We can count the same motif with a slightly different combination of multiplications: First we count the number of neurons `k`, such that `i`->`k` and `j`->`k`. Then we enforce existance of `i`->`j` with an element-wise multiplication.

Together, this assembles the same simplex motif.

In [ ]:
sp_adjacency = S.matrix.astype(bool).astype(int)
sp_adjacency_arr = sparse.coo_array(sp_adjacency)

# Counts instances of:
# i -> k  &  j -> k
ikj_count = sp_adjacency * sp_adjacency.transpose()

ikj_count_arr = sparse.coo_array(ikj_count)

# Also enfore i -> j
n_3_simplices = (ikj_count_arr * sp_adjacency_arr).sum()

print(f"Number of 3-simplices: {n_3_simplices}")

As we can see, we can search for various types of motifs fairly efficiently by using matrix multiplications of sparse matrices. Once again, the same matrix multiplications can be done on the dense representation of connectivity, but less efficiently. 

### Exercise: 
Combine the above with use of a KDTree to find the 3-cycles where all three neurons are within 150 um of each other.